# SHAP Explainability

Adding per-instance feature attribution on top of the tuned XGBoost model
from notebooks/03_model_training.ipynb. This produces the reasoning that
will populate fault_detections.shap_summary for each detected fault, and
gives the Predictive LLM agent real, grounded evidence to cite
rather than an opaque confidence score.

In [1]:
import pandas as pd
import json
import xgboost as xgb
import shap
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv("../.env")

# Load the frozen model and class order, not a retrained copy
model = xgb.XGBClassifier()
model.load_model("../models/xgboost_fault_detector.json")

with open("../models/label_encoder_classes.json") as f:
    class_names = json.load(f)

print("Model loaded. Classes:", class_names)

# Load test set - the same unseen data used for the final evaluation
user = os.environ["POSTGRES_USER"]
password = os.environ["POSTGRES_PASSWORD"]
host = os.environ["POSTGRES_HOST"]
port = os.environ["POSTGRES_PORT"]
db = os.environ["POSTGRES_DB"]
engine = create_engine(f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db}")

test_df = pd.read_sql("SELECT * FROM sensor_readings WHERE split = 'test'", engine)
feature_cols = [c for c in test_df.columns if c not in
                ("id", "ahu_id", "reading_time", "labeling", "split")]
X_test = test_df[feature_cols]

print(f"Test set: {X_test.shape}")

Model loaded. Classes: ['Normal condition', 'Return air temperature fault', 'Supply air temperature fault', 'Supply fan fault']
Test set: (35083, 18)


## Creating the explainer

Using SHAP's TreeExplainer, which computes exact SHAP values efficiently
for tree-based models like XGBoost, rather than the slower approximation
methods needed for arbitrary model types.

In [2]:
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the test set. For multi-class models, this
# returns one array of values per class, shape (n_samples, n_features, n_classes).
shap_values = explainer.shap_values(X_test)

print("SHAP values shape:", shap_values.shape)

SHAP values shape: (35083, 18, 4)


## Explaining a single prediction

Picking one row the model flagged as a fault, and showing which features
drove that specific decision. This is the format that will eventually
populate fault_detections.shap_summary for each real detected fault.

In [3]:
# Find a row genuinely predicted as Supply fan fault, to inspect
preds = model.predict(X_test)
fault_class_idx = class_names.index("Supply fan fault")

fault_rows = X_test[preds == fault_class_idx]
sample_idx = fault_rows.index[0]
sample_position = X_test.index.get_loc(sample_idx)

print(f"Explaining row at position {sample_position} (predicted: {class_names[fault_class_idx]})")
print(X_test.iloc[sample_position])

Explaining row at position 101 (predicted: Supply fan fault)
set_point_temp           22.00
return_temp              25.14
supply_air_temp          25.07
supply_fan                0.00
valve_position            0.00
heating_supply_temp_1      NaN
heating_supply_temp_2      NaN
total_heating_pump         NaN
heating_pump_1             NaN
heating_pump_2             NaN
heating_pump_3             NaN
cooling_supply_temp_1    12.96
cooling_supply_temp_2    12.74
total_cooling_pump        1.00
cooling_pump_1            0.00
cooling_pump_2            0.00
cooling_pump_3            0.50
cooling_pump_4            0.50
Name: 101, dtype: float64


In [4]:
import numpy as np

# SHAP values for this one row, for the predicted class specifically
row_shap = shap_values[sample_position, :, fault_class_idx]

contributions = pd.Series(row_shap, index=feature_cols).sort_values(key=abs, ascending=False)
print("Top feature contributions toward 'Supply fan fault' prediction:")
print(contributions.head(8))

Top feature contributions toward 'Supply fan fault' prediction:
return_temp              1.407631
set_point_temp           0.849775
total_cooling_pump       0.519705
supply_fan               0.413085
cooling_supply_temp_1    0.237495
cooling_supply_temp_2    0.216422
total_heating_pump      -0.150773
cooling_pump_3           0.133761
dtype: float32


### Interpreting the explanation

For this row (predicted Supply fan fault), the top contributing features:

| Feature | Value | SHAP contribution |
|---|---|---|
| return_temp | 25.14 | +1.41 |
| set_point_temp | 22.00 | +0.85 |
| total_cooling_pump | 1.00 | +0.52 |
| supply_fan | 0.00 | +0.41 |
| total_heating_pump | NaN | -0.15 |

The return temperature sitting 3 degrees above set point, combined with
the fan reading 0.0 while the cooling pump is actively running, tells a
coherent physical story: cooling is being prepared but not delivered,
consistent with a fan not responding to demand. total_heating_pump pushes
slightly against this prediction, correctly reflecting that the AHU is in
cooling mode, not heating mode, for this reading.

This is the kind of explanation that will be attached to each real
detected fault, giving the Predictive LLM agent concrete evidence to
reason from and cite, rather than a bare classification label.

## Building the shap_summary structure

Converting a row's SHAP values into the small JSON object that will be
stored in fault_detections.shap_summary, keeping only the top contributing
features rather than all 18, since only a handful meaningfully drive any
single prediction.

In [5]:
def build_shap_summary(row_position: int, predicted_class_idx: int, top_n: int = 5) -> dict:
    row_shap = shap_values[row_position, :, predicted_class_idx]
    contributions = pd.Series(row_shap, index=feature_cols).sort_values(key=abs, ascending=False)

    top_features = []
    for feature, value in contributions.head(top_n).items():
        top_features.append({
            "feature": feature,
            "shap_value": round(float(value), 4),
            "reading": None if pd.isna(X_test.iloc[row_position][feature])
                       else round(float(X_test.iloc[row_position][feature]), 2)
        })

    return {
        "predicted_class": class_names[predicted_class_idx],
        "top_features": top_features
    }


summary = build_shap_summary(sample_position, fault_class_idx)
print(json.dumps(summary, indent=2))

{
  "predicted_class": "Supply fan fault",
  "top_features": [
    {
      "feature": "return_temp",
      "shap_value": 1.4076,
      "reading": 25.14
    },
    {
      "feature": "set_point_temp",
      "shap_value": 0.8498,
      "reading": 22.0
    },
    {
      "feature": "total_cooling_pump",
      "shap_value": 0.5197,
      "reading": 1.0
    },
    {
      "feature": "supply_fan",
      "shap_value": 0.4131,
      "reading": 0.0
    },
    {
      "feature": "cooling_supply_temp_1",
      "shap_value": 0.2375,
      "reading": 12.96
    }
  ]
}


## Summary

- SHAP's TreeExplainer computes exact per-instance feature attributions
  for the frozen XGBoost model, loaded from models/xgboost_fault_detector.json
  rather than retrained, guaranteeing consistency with the model actually
  evaluated in notebooks/03_model_training.ipynb.
- A single explained prediction (Supply fan fault) produced a coherent,
  physically sensible explanation: elevated return temperature relative to
  set point, combined with an active cooling pump and an idle fan, matches
  the real definition of this fault rather than reading as arbitrary noise.
- build_shap_summary() converts a row's SHAP values into a compact JSON
  structure (predicted class plus top 5 contributing features with their
  values and readings), matching the shape needed for
  fault_detections.shap_summary. This function is what the Predictive
  agent will call on each newly detected fault, one row at a
  time, not on batches, since SHAP computation on a single row is fast.
